# Research: fixing VCC 2026 MSE

Scope: MSE research; all six metrics are safeguards for cell-based choices. No leaderboard uploads. Existing jobs and pre-existing edits are preserved. Code lives in this notebook; input snapshots, derived arrays, tables, and PNGs live in ignored `lib/research_mse/` and `results/research_mse/`. Each completed phase is committed separately.

Assumptions: use the pinned `cell-eval2==0.18.0` implementation and the existing 75-target H1 reference; H1 is calibration, not an untouched test. Never train on H1 for H1 predictions. Use final models only for A–C. For E1a, “full controls” means the full **generator half** on H1 (preserving the disjoint scoring half), and all available challenge controls on A–C. All improvements must be judged using the scorer's own bounded correction, not an uncapped surrogate.

## Phase 1 — exact definition and reproduction


In [ ]:
import os, sys, json, hashlib, zipfile, argparse, copy, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import anndata as ad
import scipy.sparse as sp
import importlib.metadata as metadata
import cell_eval2
V = Path.cwd()
SRC = V / 'lib/research_mse/source'
OUT = V / 'results/research_mse'
OUT.mkdir(parents=True, exist_ok=True)
REF = V / 'lib/vcc_2026_3/val_final'
RUN = V / 'lib/vcc_2026_4/runs/dev_nopie'
PKG = Path(cell_eval2.__file__).parent
sys.path.insert(0, str(V / 'code/vcc_2026_3c'))
import vcc3_generate as gen
from vcc2_common import seed_for
from cell_eval2.metrics import delta as metric_delta
from cell_eval2.moments import GroupMoments, jackknife_correction
from cell_eval2.score import _replicate_entries
from cell_eval2.scoring import score_one
assert metadata.version('cell-eval2') == '0.18.0'
wheel = Path('/tmp/research_mse_scorer/cell_eval2-0.18.0-py3-none-any.whl')
if wheel.exists():
    with zipfile.ZipFile(wheel) as z:
        mismatches = [p for p in z.namelist() if p.startswith('cell_eval2/') and p.endswith(('.py','.yaml')) and z.read(p) != (PKG.parent/p).read_bytes()]
    assert not mismatches, mismatches
    print('Installed scorer exactly matches the downloaded 0.18.0 wheel.')
files=['metrics/delta.py','moments.py','catalog.py','run.py','baseline.py','anchor.py','scoring.py','score.py','configs/vcc2026.yaml']
provenance={f:hashlib.sha256((PKG/f).read_bytes()).hexdigest() for f in files}
provenance['generator_at_start']=hashlib.sha256(Path(gen.__file__).read_bytes()).hexdigest()
(OUT/'source_hashes.json').write_text(json.dumps(provenance,indent=2))
print('Scorer:', PKG)


Let $y_{ig}$ be integer raw counts, $P_{kg}=\sum_i y_{ikg}$, and $\mu_{kg}=\log(1+50000 P_{kg}/\sum_gP_{kg})$. The log is natural. All panel genes enter library normalization. Let $G_k$ be the panel excluding **only knockdown k's own target gene**, and $w_k=1/|G_k|$.

The delete-one-cell jackknife is $J_k=(n_k-1)/n_k\sum_g\sum_i(v_{ikg}-\bar v_{kg})^2$, where $v_{ikg}=\log(1+50000(P_{kg}-y_{ikg})/(\sum_gP_{kg}-\sum_gy_{ikg}))$. It estimates sampling variance of the log-normalized group sum, not variance of a mean of cell-level logs.

For predicted cells, the first bound is $q_k=\min(J_k^{pred},J_k^{real})$. Define claim $C=\sum_kw_kq_k$ and budget $B=\sum_g\sum_{k:g\in G_k}w_k(\mu^{pred}_{kg}-\bar\mu^{pred}_{g,w})^2$, using the weighted mean over rows retaining gene g. The second bound is $r=\min(1,B/C)$ (no correction when claim is zero). The accepted prediction correction is $r q_k$.

The per-knockdown numerator is $N_k=w_k[\sum_{g\in G_k}(\mu^{pred}_{kg}-\mu^{real}_{kg})^2-J_k^{real}-rq_k]$. The uncapped diagnostic instead subtracts the full $J_k^{pred}$. The denominator is $D_k=w_k[\sum_{g\in G_k}(\mu^{real}_{kg}-\mu^{ctrl}_{g})^2-J_k^{real}-J^{ctrl}]$. **Corrections stay summed over all genes even though the own-target squared residual is removed.** There is no unique scored per-gene correction allocation in these scalar caches; per-gene diagnostics must report this explicitly.

Raw MSE is **$U=\sum_kN_k/\sum_kD_k$**, using paired finite rows. Individual numerators and denominators can be negative; they are not clipped. The summed denominator must be positive. “Capped” caps the noise correction, not errors, genes, targets, or the raw score. The real-control >5 CPM gate is for DE; it does not remove genes from expression MSE. Other panel targets remain in MSE (PDS removes all panel targets).

Scaled MSE is $\operatorname{clip}((b-U)/(b-a),0,1)$, where b is the measured generic baseline and a the split-half replicate anchor. Report the linear expression before clipping separately. Overall averages the six scaled members; MSE alone has this [0,1] clamp.


In [ ]:
# Executable, version-pinned source quotations with file and line numbers.
quotes = {
 'configs/vcc2026.yaml': [(10,26),(60,64)],
 'metrics/delta.py': [(41,43),(213,227),(280,290),(697,700),(832,835),(854,864),(878,881),(1228,1236)],
 'moments.py': [(173,180)], 'catalog.py':[(345,357)], 'run.py':[(1896,1905),(1912,1913)],
 'baseline.py':[(112,125),(167,181),(233,239),(323,327),(333,335)],
 'anchor.py':[(117,121),(135,140),(178,195),(277,287)],
 'score.py':[(394,401)]}
for file, spans in quotes.items():
    lines=(PKG/file).read_text().splitlines()
    text='\n'.join(f'{i}: {lines[i-1]}' for a,b in spans for i in range(a,min(b,len(lines))+1))
    display(Markdown(f'**`cell_eval2/{file}`**\n```python\n{text}\n```'))


The generic baseline first takes **each real non-control perturbation's mean raw counts**, then averages those means equally over perturbations. For gene g it omits the perturbation targeting g; it includes the scored perturbation for other genes. This is not a leave-one-knockdown-out baseline and is not the arithmetic mean of log-normalized pseudobulks. It rescales resampled real controls gene-wise by generic mean / control mean, with float32 fractional counts (an explicit baseline exception to submission integer requirements). Therefore its emitted pseudobulk contains sampling noise and need not equal the noise-free log of its expected counts. Genes absent in controls cannot be induced by that multiplicative construction.

The replicate anchor uses five independent seeded disjoint half-splits within each perturbation and controls, scoring half B against half A, with each side's own controls. The anchor is the mean of **five aggregate scores**, not pooled per-target values and not a Spearman–Brown-corrected MSE. The H1 frozen bundle is the authority for its exact seeds and settings.


In [ ]:
genes=pd.read_csv(V/'data/gene_names.csv')['gene_name'].astype(str).to_numpy()
G=len(genes); gp={g:i for i,g in enumerate(genes)}
z=np.load(next((REF/'real_cache').glob('*.moments.npz')))
print('Reference cache fields:', z.files)
perts=z['perts'].astype(str); means=z['means'].astype(float); jk=z['jk'].astype(float)
targets=sorted(set(perts)-{'non-targeting'}); T=len(targets); rows=np.arange(T)
ridx={p:i for i,p in enumerate(perts)}; ri=np.array([ridx[t] for t in targets]); ci=ridx['non-targeting']
R=means[ri]; ctrl=means[ci]; Jr=jk[ri]; Jc=jk[ci]
own=np.array([gp.get(t,-1) for t in targets]); nrow=np.where(own>=0,G-1,G); weights=1/nrow
mask=np.ones((T,G),bool); mask[rows[own>=0],own[own>=0]]=False
Dden=(((R-ctrl)**2*mask).sum(1)-Jr-Jc)/nrow
base=pd.read_csv(REF/'bundle/baseline_agg.csv').set_index('statistic').loc['mean'].to_dict()
anchor=pd.read_parquet(REF/'bundle/anchor_agg.parquet')
entries=_replicate_entries(base,anchor); MSE='expr_mse_unbiased_capped_norm'
b=float(base[MSE]); a=float(anchor.set_index('metric').loc[MSE,'replicate'])
def scaled_mse(raw): return (b-raw)/(b-a)
def tidy(path):
    df=pd.read_csv(path); df.columns=['perturbation','metric','value']
    return df.pivot(index='perturbation',columns='metric',values='value').loc[targets]
bl=tidy(REF/'baseline/baseline_results.csv')
np.testing.assert_allclose(Dden,bl.expr_distance_unbiased,rtol=2e-12,atol=1e-14)
print(f'{T} targets × {G} genes; denominator sum={Dden.sum():.12g}; baseline={b:.12g}; replicate={a:.12g}')
manifest=json.loads((REF/'bundle/manifest.json').read_text())
print('Replicate seeds:',manifest['derived_seeds'])
arm_paths={
 'arm1':SRC/'h1_bench_v5a1_p1/a1_arm1_s1_floor',
 'arm2':SRC/'h1_bench_v5a1_p1/a1_arm2_s1',
 'arm7':SRC/'h1_bench_v5a1_p3/a1_arm7_s2_floor'}
phase1_rows=[]; phase1_summary=[]
for name,path in arm_paths.items():
    df=tidy(path/'results.csv')
    np.testing.assert_allclose(df.expr_distance_unbiased,Dden,rtol=2e-12,atol=1e-14)
    raw=df.expr_mse_unbiased_capped.sum()/Dden.sum()
    reported=pd.read_csv(path/'agg_results.csv').set_index('statistic').loc['mean',MSE]
    np.testing.assert_allclose(raw,reported,rtol=2e-12)
    part=pd.DataFrame({'arm':name,'target':targets,'normalizer':Dden,
       'uncapped_error':df.expr_mse_unbiased.to_numpy(),'capped_error':df.expr_mse_unbiased_capped.to_numpy()})
    part['correction_withheld']=part.capped_error-part.uncapped_error
    phase1_rows.append(part)
    phase1_summary.append({'arm':name,'raw_capped':raw,'raw_uncapped':df.expr_mse_unbiased.sum()/Dden.sum(),
                          'mse_unclamped':scaled_mse(raw),'mse_scaled':np.clip(scaled_mse(raw),0,1),
                          'row_reaggregation_difference':raw-reported})
phase1_rows=pd.concat(phase1_rows,ignore_index=True)
phase1_rows.to_csv(OUT/'phase1_per_target.csv',index=False)
display(pd.DataFrame(phase1_summary))
display(phase1_rows)
print('This table independently reconstructs denominators and aggregate ratios from the saved scorer per-target numerators. Phase 2 also regenerates cells and independently rebuilds those numerators.')


In [ ]:
# Exact metric audit implementation, cross-checked against official functions below.
excl={i:int(j) for i,j in enumerate(own) if j>=0}
real_moments=GroupMoments(perts,np.asarray(z['counts']),np.asarray(z['sumsq']),jk)
def exact_mse(P,Jpred=None,verify=False):
    P=np.asarray(P,float); Jpred=np.zeros(T) if Jpred is None else np.asarray(Jpred,float)
    plugin=((P-R)**2*mask).sum(1)/nrow
    claim_rows=np.minimum(Jpred,Jr)
    claim=float((claim_rows/nrow).sum())
    budget=metric_delta._across_pert_budget(P,excl,weights)
    ratio=min(1.0,budget/claim) if claim>0 else 1.0
    uncapped=plugin-(Jr+Jpred)/nrow
    capped=plugin-(Jr+ratio*claim_rows)/nrow
    if verify:
        pm=GroupMoments(np.array(targets),np.full(T,400),np.zeros(T),Jpred)
        kw=dict(pred_bulk=(np.array(targets),P),real_bulk=(perts,means),pred_moments=pm,
                real_moments=real_moments,control='non-targeting',genes=genes,comparator='bulk_lognorm')
        official=metric_delta.mse_unbiased_capped(**kw)
        np.testing.assert_allclose(capped,[official[t] for t in targets],rtol=1e-10,atol=1e-13)
    return dict(raw=capped.sum()/Dden.sum(),raw_uncapped=uncapped.sum()/Dden.sum(),
                plugin=plugin,capped=capped,uncapped=uncapped,budget=budget,claim=claim,ratio=ratio,
                prediction_correction=ratio*claim_rows/nrow)
# Different prediction spreads and nonzero claims exercise BOTH correction bounds.
for s in [0,0.1,1,2]:
    exact_mse(ctrl+s*(R-R.mean(0)),Jr*1.5,verify=True)
print('Independent implementation agrees with official kernels, including both correction bounds.')


In [ ]:
# Load generator controls without reading perturbed cells into memory.
import pickle
control_cache=V/'lib/research_mse/h1_generator_controls.pkl'
if control_cache.exists():
    with control_cache.open('rb') as fh: line,train_rows,score_rows=pickle.load(fh)
else:
    h=ad.read_h5ad(V/'lib/h1/H1_full_2025.h5ad',backed='r')
    try:
        line, train_rows, score_rows=gen.h1_controls(h,h.obs['target_gene'].astype(str).to_numpy(),list(genes),2026,'guide_id')
    finally: h.file.close()
    with control_cache.open('wb') as fh: pickle.dump((line,train_rows,score_rows),fh,protocol=5)
settings=json.loads((RUN/'h1_bench_g3/selected_generator.json').read_text())['settings']
cfg=gen.Cfg(argparse.Namespace(**dict(settings,detect_floor=False)))
pdir=RUN/'h1_pred_g3'
pred=np.load(pdir/'raw_predictions.npz')
t300=np.load(pdir/'targets.npy').astype(str); pos={t:i for i,t in enumerate(t300)}; idx=np.array([pos[t] for t in targets])
delta=pred['delta'].astype(float); gcols=pred['gene_cols']; run_set=np.isin(np.arange(G),gcols)
kd=gen.kd_residual(gen.prediction_ontarget(pdir,targets),cfg)
def delta_panel(s=1,beta=1,extension=None):
    dbar=delta.mean(0,keepdims=True)
    D=np.zeros((T,G),np.float32)
    if extension is not None: D[:,~run_set]=np.asarray(extension)[~run_set]
    D[:,gcols]=(beta*dbar+s*(delta-dbar))[idx]
    return D

def bulk_from_calibrated(D):
    P=np.empty((T,G)); factors=np.empty_like(P)
    for i in rows:
        factors[i],share,_=gen.project_factors(line.share,D[i],int(own[i]) if own[i]>=0 else None,kd[i])
        P[i]=np.log1p(50000*share)
    return P,factors

def intended(s=1,beta=1,floor=False,D=None):
    local=copy.copy(cfg); local.detect_floor=floor
    Dc=gen.calibrate(delta_panel(s,beta) if D is None else D,line,targets,local)
    P,F=bulk_from_calibrated(Dc)
    return P,Dc,F
P2,D2,F2=intended()
P1,D1,F1=intended(floor=True)
P7,D7,F7=intended(s=2,floor=True)
print('Control cells:',line.n,'generator;',line.n_reference,'scoring. Model genes:',len(gcols))
for name,P in [('arm1',P1),('arm2',P2),('arm7',P7)]:
    sc=exact_mse(P,verify=True); print(name,'noise-free raw:',sc['raw'],'linear:',scaled_mse(sc['raw']))


In [ ]:
# Reproduce the ORIGINAL A6 terciles; no averaging of tercile ratios.
old=json.loads((SRC/'h1_decomp/results.json').read_text())
size=np.sqrt(np.maximum(((R-R.mean(0))**2).sum(1)-Jr,0))
terc=np.argsort(np.argsort(size))*3//T
num=exact_mse(P2)['capped']; terc_rows=[]
for q,name in enumerate(['weak','mid','strong']):
    sel=terc==q
    terc_rows.append(dict(tercile=name,n=int(sel.sum()),denominator_sum=Dden[sel].sum(),
       denominator_weight=Dden[sel].sum()/Dden.sum(),model_ratio=num[sel].sum()/Dden[sel].sum(),
       baseline_ratio=bl.expr_mse_unbiased_capped.to_numpy()[sel].sum()/Dden[sel].sum()))
terc_table=pd.DataFrame(terc_rows)
display(terc_table)
print('Weighted model:',(terc_table.model_ratio*terc_table.denominator_weight).sum(),
      'baseline:',(terc_table.baseline_ratio*terc_table.denominator_weight).sum())
print('Unweighted tercile model average:',terc_table.model_ratio.mean())
print('A6 sign correction: 0.974 is ABOVE baseline 0.9535, hence negative unclamped skill.')
terc_table.to_csv(OUT/'phase1_terciles.csv',index=False)
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
terc_table.plot.bar(x='tercile',y=['model_ratio','baseline_ratio'],ax=ax[0],rot=0)
terc_table.plot.bar(x='tercile',y='denominator_weight',ax=ax[1],rot=0,legend=False)
ax[0].set_ylabel('Raw MSE within tercile');ax[1].set_ylabel('Weight in full-panel ratio')
fig.tight_layout();fig.savefig(OUT/'phase1_terciles.png',dpi=160);plt.show()


### How to win MSE under this formula

Improve the **sum of absolute squared pseudobulk residuals**, in natural-log bulk space. A large residual in any gene or knockdown remains costly: neither cap limits its damage. Strong-response targets carry most denominator mass, but every target has nearly the same numerator weight (1/18,532 here). Weak targets can have huge individual error/strength ratios while contributing little to the aggregate; optimizing the average of those ratios is the wrong objective.

The shared response is valuable only to the extent its corrected profile reduces that same error sum. The baseline gets to use the target line's real perturbations; learning generic responses from other lines is a genuine prediction problem. Genes outside the model's 5,000 still count. The >5 CPM gate does not protect MSE from errors elsewhere, and composition normalization can move genes assigned zero log2FC.

Generator noise is partly corrected, not automatically removed. Both real and predicted sampling terms enter, but the prediction deduction cannot exceed the real row's jackknife or the panel's own variation budget. Common base-cell errors shared across all knockdowns do not earn across-knockdown budget. Full-control total matching should therefore be tested with these bounds and all six metrics. Do not infer a gain from a two-seed noise reduction alone.

The floor is an especially direct intervention: it turns small predictions into larger shifts regardless of MSE evidence. The current implementation uses a CPM mean/variance z approximation, **not simulated scorer Wilcoxon + BH**; unreachable shifts are assigned 8. Test skipping costly floor changes before broadening model complexity.

Corrections to Part A established from code: raw errors are not capped; both sides have sampling corrections; MSE has no DE CPM gate; common base cells already exist; the current floor still has the unreachable→8 behavior; the baseline is not simply the arithmetic mean of log pseudobulks; and 0.974 is worse, not better, than 0.9535. H1 uses a disjoint control split, not all H1 controls for generation.

Status: phase 1 numerical results above reproduce cached per-target scores and aggregate ratios; independent regeneration of the cells is the first phase 2 check. No submission or existing generator has been modified.


### Model and submission provenance audit

`vcc_2026_2.ipynb`, opening results table and lesson 4, identifies **+0.145 as the improvement from −0.054 to +0.091**. The documented best Model 2 submission is `vcc_2026_3.ipynb`'s `model2_top2000_floor`: `re`, top 2,000 by |Δ|/SE, alpha 1, detection z=5, thinning generator. No separate +0.145 overall is substantiated by the available records.

Model 2 reads the older `lib/sources/<source>/effects_panel.h5ad` tables (`code/vcc_2026_3/vcc2_meta.py:51`), with DerSimonian–Laird per-gene tau² pooled over targets (`:73`) and precision combination (`:100`). It enforces one background vote; `vcc3_ridge.py:66` restricts T-cell evidence to Stim48hr. Model 3's PCA uses uncentered effects and correction ridges; its program-only model is not used here.

`code/transformer/model.py` and `losses.py` match the supplied architecture and loss description, with one nuance: PIE **evidence** is off in dev_nopie, but `gene_features="pie"` still uses PIE gene embeddings, including coverage flags. The saved dev config sets a maximum of 20 epochs and patience 3, not a fixed 12 epochs; the verified best checkpoint is epoch 12, gamma 0.0491102. The recorded prediction checks explicitly show H1 absent from loaded sources, targets, and evidence slots. The effect-bank inputs are `lib/effects_v4`; the panel baseline construction is in `scripts/build_baseline_panel.py`.

Numerical resolution of the tercile puzzle: weak/mid/strong denominator weights are **1.7567% / 11.7402% / 86.5031%**. Their model ratios 1.071720 / 1.114949 / 0.953265 combine to **0.974328**; the baseline combines to **0.953469**. An unweighted tercile average gives 1.046645 and is not the metric.


## Phase 2 — anatomy of the H1 error

Regenerate arms 1, 2, and 7 with the recorded guide-stratified control pool and the original maximum-pool-size rule. Compute the official jackknife on each generated count matrix and apply both caps jointly over the whole 75-target panel. Counts are streamed one target at a time; only pseudobulks and jackknives are cached.

For exact additive mechanism attribution, write the residual as shared + specific + floor + generator. Allocate their cross terms symmetrically: component v receives `sum(v * total_residual)` (its own square plus half of every pairwise interaction). Also report standalone squared magnitudes and actual counterfactual changes; these answer different questions. Separate the per-target mean-over-scored-genes offset orthogonally. Outside-model genes, CPM deciles, and strength terciles are overlapping diagnostic partitions, not extra independent mechanisms. The official scalar noise deductions remain explicit row-level terms; do not invent a gene-specific correction from a scalar cache.


In [ ]:
# Phase 2: regeneration and independent metric reproduction.
max_pool=int(max(400,z['counts'][ri].max()))
pool_cfg=copy.copy(cfg);pool_cfg.n_cells=max_pool
common=gen.pick_cells(line,np.zeros(G),pool_cfg,np.random.default_rng(seed_for(2026,'H1','base_controls')))
print('Common base pool:',max_pool,'; first 400 cells used for every target.',flush=True)
def generated_bulk(name,Dc,seed=2026,mode='original',save_cells=False):
    cache=OUT/f'{name}_seed{seed}_{mode}.npz'
    if cache.exists():
        zz=np.load(cache);return {k:zz[k] for k in zz.files}
    local=copy.copy(cfg);local.seed=seed
    pc=copy.copy(pool_cfg)
    picked=gen.pick_cells(line,np.zeros(G),pc,np.random.default_rng(seed_for(seed,'H1','base_controls')))[:400]
    P=np.empty((T,G));J=np.empty(T);mx=np.empty(T);start=time.time()
    for i,t in enumerate(targets):
        X=gen.generate_one(line,Dc[i],int(own[i]),kd[i],local,pick=picked,line_key='H1',target_key=t)[0]
        counts=np.asarray(X.sum(0)).ravel().astype(float)
        P[i]=np.log1p(50000*counts/counts.sum())
        J[i]=jackknife_correction(X,np.zeros(X.shape[0],np.int32),1,50000,chunk=32)[0]
        mx[i]=np.asarray(X.sum(1)).max()
        if (i+1)%10==0: print(name,i+1,'/',T,'elapsed',round(time.time()-start,1),flush=True)
    result=dict(P=P,J=J,max_cell_counts=mx,picked=picked)
    np.savez_compressed(cache,**result)
    return result
regenerated={}
checks=[]
for name,Dc in [('arm1',D1),('arm2',D2),('arm7',D7)]:
    data=generated_bulk(name,Dc);regenerated[name]=data
    sc=exact_mse(data['P'],data['J'],verify=True);saved=tidy(arm_paths[name]/'results.csv')
    err=np.abs(sc['capped']-saved.expr_mse_unbiased_capped.to_numpy()).max()
    uncerr=np.abs(sc['uncapped']-saved.expr_mse_unbiased.to_numpy()).max()
    checks.append(dict(arm=name,raw_mse=sc['raw'],unclamped=scaled_mse(sc['raw']),
      budget_ratio=sc['ratio'],claim=sc['claim'],budget=sc['budget'],max_row_error=err,
      max_uncapped_row_error=uncerr,mean_Jpred=data['J'].mean(),mean_Jreal=Jr.mean()))
    print(checks[-1],flush=True)
    np.testing.assert_allclose(sc['capped'],saved.expr_mse_unbiased_capped.to_numpy(),atol=3e-8,rtol=3e-5)
checks=pd.DataFrame(checks);checks.to_csv(OUT/'phase2_regeneration_checks.csv',index=False);display(checks)


In [ ]:
# Signed, additive mechanism decomposition; original/raw/capped totals all retained.
P7_no,D7_no,F7_no=intended(s=2,floor=False)
mechanism_rows=[]; slice_rows=[]; gene_tables={}; floors=[]
expr_bin=pd.qcut(pd.Series(line.cpm).rank(method='first'),10,labels=False).to_numpy()
for name,P0,Pi,D0,Df in [('arm2',P2,P2,D2,D2),('arm7',P7_no,P7,D7_no,D7)]:
    Pg=regenerated[name]['P'];Jp=regenerated[name]['J'];sc=exact_mse(Pg,Jp)
    shared=np.broadcast_to(P0.mean(0)-R.mean(0),R.shape).copy()
    specific=(P0-P0.mean(0))-(R-R.mean(0))
    vv={'shared_profile':shared,'knockdown_specific':specific,'floor':Pi-P0,'generator':Pg-Pi}
    residual=Pg-R;np.testing.assert_allclose(sum(vv.values()),residual,atol=1e-14)
    offset=(residual*mask).sum(1)/nrow
    residual_centered=(residual-offset[:,None])*mask
    plugin_sum=float(sc['plugin'].sum());capped_sum=float(sc['capped'].sum())
    for component,v in vv.items():
        allocation=float((v*residual*mask/nrow[:,None]).sum())
        own_square=float((v*v*mask/nrow[:,None]).sum())
        v_offset=(v*mask).sum(1)/nrow
        centered_allocation=float(((v-v_offset[:,None])*residual_centered/nrow[:,None]).sum())
        mechanism_rows.append(dict(arm=name,component=component,signed_contribution=allocation,
           standalone_square=own_square,plugin_share=allocation/plugin_sum,capped_share=allocation/capped_sum,
           raw_metric_contribution=allocation/Dden.sum(),after_offset_contribution=centered_allocation))
    for component,value in [('real_noise_deduction',-float((Jr/nrow).sum())),
                            ('pred_noise_capped_deduction',-float(sc['prediction_correction'].sum()))]:
        mechanism_rows.append(dict(arm=name,component=component,signed_contribution=value,
            standalone_square=np.nan,plugin_share=value/plugin_sum,capped_share=value/capped_sum,
            raw_metric_contribution=value/Dden.sum(),after_offset_contribution=np.nan))
    print(name,'plugin / uncapped / capped:',plugin_sum/Dden.sum(),sc['raw_uncapped'],sc['raw'],
          'offset share of plugin:',float((offset**2).sum()/plugin_sum))
    print(name,'noise-free counterfactual floor raw delta:',exact_mse(Pi)['raw']-exact_mse(P0)['raw'],
          'generator raw delta (including accepted prediction correction):',sc['raw']-exact_mse(Pi)['raw'])
    gene_err=(residual**2*mask/nrow[:,None]).sum(0)
    gene_table=pd.DataFrame({'gene':genes,'control_cpm':line.cpm,'in_model':run_set,
       'plugin_contribution':gene_err,'plugin_share':gene_err/plugin_sum,
       'raw_metric_contribution':gene_err/Dden.sum(),
       'floor_signed_contribution':((Pi-P0)*residual*mask/nrow[:,None]).sum(0),
       'generator_signed_contribution':((Pg-Pi)*residual*mask/nrow[:,None]).sum(0)})
    gene_table=gene_table.sort_values('plugin_contribution',ascending=False);gene_tables[name]=gene_table
    gene_table.head(50).to_csv(OUT/f'phase2_{name}_top50_genes.csv',index=False)
    display(gene_table.head(50))
    for q in range(10):
        use=expr_bin==q
        slice_rows.append(dict(arm=name,partition='expression_decile',group=str(q+1),
             n=int(use.sum()),plugin_sum=gene_err[use].sum(),plugin_share=gene_err[use].sum()/plugin_sum,
             real_correction=np.nan,pred_correction=np.nan,capped_raw=np.nan))
    for label,use in [('in_model',run_set),('outside_model',~run_set)]:
        slice_rows.append(dict(arm=name,partition='model_gene_set',group=label,n=int(use.sum()),
             plugin_sum=gene_err[use].sum(),plugin_share=gene_err[use].sum()/plugin_sum,
             real_correction=np.nan,pred_correction=np.nan,capped_raw=np.nan))
    for q,label in enumerate(['weak','mid','strong']):
        use=terc==q
        slice_rows.append(dict(arm=name,partition='strength_tercile',group=label,n=int(use.sum()),
          plugin_sum=sc['plugin'][use].sum(),plugin_share=sc['plugin'][use].sum()/plugin_sum,
          real_correction=-(Jr/nrow)[use].sum(),pred_correction=-sc['prediction_correction'][use].sum(),
          capped_raw=sc['capped'][use].sum()/Dden[use].sum()))
    raised=(np.abs(Df)>np.abs(D0)+1e-6)&mask
    increments=(np.abs(Df)-np.abs(D0))[raised]
    floor_sq=(Pi-P0)**2*mask/nrow[:,None]
    floor_error_share=(residual**2*raised/nrow[:,None]).sum()/plugin_sum
    print(name,'floor-raised entries',int(raised.sum()),'increment quantiles',
          np.quantile(increments,[0,.25,.5,.75,.9,.99,1]) if len(increments) else [],
          'share of plugin error on raised genes',floor_error_share)
    if name=='arm7':
        for q in range(10):
            use=raised&(expr_bin[None,:]==q)
            floors.append(dict(decile=q+1,raised=int(use.sum()),
              median_delta=float(np.median(np.abs(Df)[use])) if use.any() else np.nan,
              max_delta=float(np.abs(Df)[use].max()) if use.any() else np.nan,
              n_abs_delta_ge3=int((use&(np.abs(Df)>=3)).sum()),
              n_abs_delta_ge7_99=int((use&(np.abs(Df)>=7.99)).sum()),
              plugin_error_share=float((residual**2*use/nrow[:,None]).sum()/plugin_sum),
              standalone_floor_square=float(floor_sq[use].sum())))
        floor_entries=pd.DataFrame({'target':np.repeat(targets,G)[raised.ravel()],
          'gene':np.tile(genes,T)[raised.ravel()], 'cpm':np.broadcast_to(line.cpm,R.shape)[raised],
          'before':D0[raised],'after':Df[raised],'increase':increments})
        floor_entries.nlargest(50,'increase').to_csv(OUT/'phase2_largest_floor_increases.csv',index=False)
        display(floor_entries.nlargest(20,'increase'))
mechanisms=pd.DataFrame(mechanism_rows);slices=pd.DataFrame(slice_rows);floor_table=pd.DataFrame(floors)
mechanisms.to_csv(OUT/'phase2_mechanisms.csv',index=False)
slices.to_csv(OUT/'phase2_partitions.csv',index=False);floor_table.to_csv(OUT/'phase2_floor_deciles.csv',index=False)
display(mechanisms);display(slices);display(floor_table)
for name in ['arm2','arm7']:
    np.testing.assert_allclose(mechanisms.loc[mechanisms.arm==name,'raw_metric_contribution'].sum(),
                              exact_mse(regenerated[name]['P'],regenerated[name]['J'])['raw'],atol=1e-12)
fig,axes=plt.subplots(1,3,figsize=(15,4))
mechanisms.pivot(index='component',columns='arm',values='raw_metric_contribution').plot.barh(ax=axes[0])
slices[slices.partition=='expression_decile'].pivot(index='group',columns='arm',values='plugin_share').reindex([str(i) for i in range(1,11)]).plot.bar(ax=axes[1],rot=0)
axes[1].set_xlabel('Control CPM decile');axes[1].set_ylabel('Share of squared residual')
floor_table.plot.bar(x='decile',y='raised',ax=axes[2],rot=0,legend=False)
axes[2].set_ylabel('Target × gene entries raised by floor')
fig.tight_layout();fig.savefig(OUT/'phase2_anatomy.png',dpi=160);plt.show()


### Phase 2 findings

Independent regenerated-cell numerators reproduce the original scorer per target to **2.4×10⁻¹⁶ or better**, for arms 1, 2 and 7. The phase 1 cached-table reproduction is now also verified from integer counts and recomputed jackknives.

For arm 7, additive signed shares of the uncorrected squared residual are **shared 2.35%, specific 57.11%, floor 33.44%, generator 7.09%**. The scorer then deducts real noise (0.07472 raw units) and accepted prediction noise (0.06267). Removing the floor at fixed s=2 lowers noise-free raw MSE by **0.71024**; generating cells adds back **0.07331** after the scorer's correction. For arm 2, generation adds **0.09057** after correction. Its across-target budget accepts only 69.25% of the already row-capped claim; the full predicted jackknife would deduct 0.14700 raw units, but only 0.04322 is accepted. Noise is therefore materially penalized.

Arm 7's floor raises **341,777 target×gene entries**; the median imposed increase is **0.1341 log2FC**, 99th percentile 0.4378, maximum 1.0049. No retained raised entry reaches |Δ|≥3 or ±8 (the largest resulting |Δ| is 1.0072). The unreachable→8 implementation exists but **does not explain these H1 arms**. Broad, modest changes in expressed genes cause the damage; caps 2 and 3 cannot help this dataset.

Genes outside the model account for **40.55%** of arm 7's squared residual and **65.35%** of arm 2's. The per-target global-offset component is only **0.739% / 0.239%** respectively; E4 has a small oracle ceiling. The top-gene and partition tables above retain the scalar correction terms separately, because a scalar cache cannot uniquely attribute them to genes.

Additional provenance correction: the transformer's `basal_table` prefers `*_baseline_v2.csv` when present. Those features normalize to the **6,013 shared genes** (`scripts/build_baseline_v2.py`), with a separate detection-missing flag; panel-only baselines are a fallback. This differs from Part A's simpler baseline-panel description and is preserved rather than silently changing the model inputs.


## Phase 3 — controlled experiments

Screen noise-free pseudobulks before spending time on cell generation. Each primary screen changes one ingredient relative to arm 7; additional no-floor rows are explicitly labeled diagnostic. Select cell-based candidates using all six metrics, with unclamped MSE retained. H1-based oracle results are excluded from candidates.

E1c is the existing generator's default (same base pool and per-gene random streams across knockdowns), so its useful control is independent base selection. E2a's requested rule means **skip** the floor when its required absolute log2FC exceeds the cap; it does not truncate an unreachable floor to the cap. E2b with a constant X is identical to E2a; here E2b instead uses a predeclared CPM-dependent threshold (0.5 / 1 / 2 over the lowest 3 / next 3 / highest 4 deciles).


In [ ]:
from cell_eval2.metrics.discrimination import discrimination_score
screen=[]; candidates={}
def screen_arm(name,Dc,kind,oracle=False):
    P,F=bulk_from_calibrated(Dc);sc=exact_mse(P)
    pdsv=discrimination_score(pred_bulk=(np.array(targets),P),real_bulk=(perts,means),
       distance='cosine',rank_denominator='n-1',tie_policy='midrank',exclude_target_gene=True,
       exclusion_scope='panel',control_source='real',genes=genes,control='non-targeting')
    pds=float(np.mean([pdsv[t] for t in targets]));entry=entries['pds_cosine']
    row=dict(arm=name,family=kind,oracle=oracle,mse_raw=sc['raw'],mse_unclamped=scaled_mse(sc['raw']),
       mse_scaled=np.clip(scaled_mse(sc['raw']),0,1),pds_raw=pds,pds_scaled=score_one(pds,entry.base,entry.scoring))
    screen.append(row);candidates[name]=dict(D=Dc,P=P,F=F);return row
screen_arm('arm7',D7,'reference');screen_arm('arm2',D2,'reference')
for s in [0,0.25,0.5,0.75,1,1.5,2,3]:
    P,D,F=intended(s=s)
    screen_arm(f'E5_global_s{s:g}_nofloor',D,'E5 global size')
up,down=gen.detection_floor(line,cfg)
def capped_floor(D0,limit):
    required=np.where(D0>0,up[None,:],down[None,:])
    valid=(D0!=0)&(required<=np.asarray(limit))
    D=np.where(valid,np.sign(D0)*np.maximum(np.abs(D0),required),D0)
    D[:,~line.gate]=0;D[rows,own]=0
    return D.astype(np.float32)
for cap in [0.5,1,2,3]:
    screen_arm(f'E2a_skip_above_{cap:g}',capped_floor(D7_no,cap),'E2a floor cap')
limits=np.where(expr_bin<3,0.5,np.where(expr_bin<6,1.,2.))
screen_arm('E2b_cpm_thresholds',capped_floor(D7_no,limits[None,:]),'E2b floor by CPM')
# Shared extension: equal background weights, non-H1 sources; each CD4 condition is a separate source
# as explicitly requested for this experiment (unlike Model 2's one-CD4-condition rule).
sys.path.insert(0,str(SRC))
from v5lib import source_line_means
source_names=['K562','RPE1','HepG2','Jurkat','HCT116','HEK293T','CD4_Rest','CD4_Stim8hr','CD4_Stim48hr']
extfile=OUT/'shared_extension_nonh1.npz'
if extfile.exists(): extension=np.load(extfile)['mean']
else:
    extension,per_source,source_counts=source_line_means(source_names,genes)
    np.savez_compressed(extfile,mean=extension,per_source=per_source,source_counts=source_counts)
for beta in [1,1.5,2,2.5,3]:
    _,D,_=intended(s=2,beta=beta,floor=True)
    screen_arm(f'E3a_beta{beta:g}_floor',D,'E3a shared size')
    for s in [1,2]:
        _,D,_=intended(s=s,beta=beta)
        screen_arm(f'E3a_beta{beta:g}_s{s:g}_nofloor',D,'E3a no-floor diagnostic')
    D=delta_panel(s=2,beta=beta,extension=beta*extension)
    _,D,_=intended(D=D,floor=True)
    screen_arm(f'E3b_beta{beta:g}_extension_floor',D,'E3b shared extension')
# Oracle in scorer space only. Correct real noise covariance induced by reusing its estimated mean.
oracleP=R.mean(0)+(P7_no-P7_no.mean(0))
oracle_plugin=((oracleP-R)**2*mask).sum(1)/nrow
oracle_J=(1-2/T)*Jr+Jr.sum()/T**2
oracle_raw=(oracle_plugin-oracle_J/nrow).sum()/Dden.sum()
print('E3c oracle shared-profile raw MSE, covariance-adjusted approximation:',oracle_raw,
      'linear skill:',scaled_mse(oracle_raw),'(upper bound; never a candidate)')
print('Oracle target-exclusion covariance correction uses full-gene scalar jackknives, matching the scorer approximation.')
screen_df=pd.DataFrame(screen).sort_values('mse_raw');screen_df.to_csv(OUT/'phase3_pseudobulk_screen.csv',index=False)
display(screen_df)
fig,ax=plt.subplots(figsize=(10,6))
for label,group in screen_df.groupby('family'):ax.scatter(group.mse_unclamped,group.pds_scaled,label=label)
ax.axvline(0,color='black',lw=1);ax.set_xlabel('Unclamped scaled MSE (noise-free)');ax.set_ylabel('Scaled PDS')
ax.legend(fontsize=8);fig.tight_layout();fig.savefig(OUT/'phase3_pseudobulk_screen.png',dpi=160);plt.show()


In [ ]:
# Reuse the exact pdex MWU kernel, scorer normalization, LFC/BH, and metric dispatch.
# The only optimization is retaining the same reference column index between targets/arms.
import polars as pl
import numba
from numba_mwu import sparse_column_index
from pdex._math import mwu
from cell_eval2.de_compute import _to_linear,_log1p_view,_lfc_from_means,_bh_per_target,_grouped_means
from cell_eval2.de import prepare_de
from cell_eval2.run import dispatch_de_metrics,aggregate_metrics
from cell_eval2.config import EvalConfig
numba.set_num_threads(2)
score_cfg=EvalConfig.from_yaml(REF/'bundle/config.yaml')
metric_names=list(gen.VCC2026_METRICS)
# Read only X of reference controls, avoiding copies of unrelated H1 layers.
h=ad.read_h5ad(V/'lib/h1/H1_full_2025.h5ad',backed='r')
try:
    small=ad.AnnData(X=h.X[np.sort(score_rows)],var=h.var.copy())
    control_X=gen.aligned_X(small,list(genes));del small
finally:h.file.close()
control_ad=ad.AnnData(control_X,var=pd.DataFrame(index=genes))
control_linear=_to_linear(control_ad,'counts',1e6);del control_ad
ref_mean=_grouped_means(control_linear.X,np.arange(control_linear.n_obs),np.array([0,control_linear.n_obs]),1,log_space=False)[0]
de_gate=ref_mean>5
assert np.array_equal(de_gate,line.gate)
reference_log=_log1p_view(control_linear)
ref_index=sparse_column_index(reference_log.X[:,de_gate].tocsr())
del reference_log,control_linear
real_de=pl.read_parquet(arm_paths['arm7']/'de_real.parquet')
def predicted_de(X,target):
    lin=_to_linear(ad.AnnData(X,var=pd.DataFrame(index=genes)),'counts',1e6)
    mean=_grouped_means(lin.X,np.arange(X.shape[0]),np.array([0,X.shape[0]]),1,log_space=False)[0]
    log=_log1p_view(lin)
    p=np.clip(np.asarray(mwu(log.X[:,de_gate].tocsr(),ref_index).pvalue),0,1)
    lfc=_lfc_from_means({target:mean[de_gate],'non-targeting':ref_mean[de_gate]},genes[de_gate],reference='non-targeting',epsilon=1e-9)
    return _bh_per_target(lfc.with_columns(pl.Series('p_value',p),pl.Series('p_adj',np.ones(len(p)))))
# Validate the reused-kernel route against the original official run before accepting any new result.
X=gen.generate_one(line,D2[0],int(own[0]),kd[0],cfg,pick=common[:400],line_key='H1',target_key=targets[0])[0]
check_de=predicted_de(X,targets[0]).to_pandas().set_index('feature').sort_index()
old_de=pl.read_parquet(arm_paths['arm2']/'de_pred.parquet').filter(pl.col('target')==targets[0]).to_pandas().set_index('feature').sort_index()
assert check_de.index.equals(old_de.index)
for col in ['p_value','p_adj','log2_fold_change']:
    print('SAVED_DIFFERENCE',col,float(np.max(np.abs(check_de[col]-old_de[col]))),flush=True)
    print('Official saved DE agreement',col,float(np.nanmax(np.abs(check_de[col]-old_de[col]))),flush=True)

# Compare the optimization to an unmodified, current official compute_de call.
# This checks preprocessing, reference handling, BH, and feature alignment independently.
from cell_eval2.de_compute import compute_de
joined=ad.AnnData(sp.vstack([X,control_X],format='csr'),
    obs=pd.DataFrame({'target':[targets[0]]*400+['non-targeting']*control_X.shape[0]},index=[f'check{i}' for i in range(400+control_X.shape[0])]),
    var=pd.DataFrame(index=genes))
official=compute_de(joined,groupby='target',reference='non-targeting',input_type='counts',backend='pdex',threads=2,
    mean_calc='arithmetic',epsilon=1e-9,clip_value=None,fdr_scope='per_pert',target_sum=1e6,filter_gene_min_cpm_cell=5)
official=official.to_pandas().set_index('feature').sort_index()
for col in ['p_value','p_adj','log2_fold_change']:
    print('CURRENT_OFFICIAL_DIFFERENCE',col,float(np.max(np.abs(check_de[col]-official[col]))),flush=True)
    np.testing.assert_allclose(check_de[col],official[col],rtol=1e-10,atol=1e-12)
del joined,official


In [ ]:
from vcc_exact_totals import pin_totals
full_mean=np.asarray(line.X.sum(axis=0,dtype=np.float64)).ravel()/line.n
cell_results=[]
short_names={'pds_cosine':'pds','expr_mse_unbiased_capped_norm':'mse','de_wilcoxon_lfc_nmae':'nmae',
 'de_wilcoxon_direction_fidelity_yield_raw':'fid','de_wilcoxon_direction_reach_raw':'reach','de_wilcoxon_sig_jaccard':'jac'}
def progress(text):
    print(text,flush=True)
    with (V/'logs/research_mse/cell_experiments.log').open('a') as f:f.write(str(text)+'\n')
def six_scores(P,J,pred_de):
    prepared=prepare_de(pred_de,real_de,control='non-targeting',sort_by='abs_log2_fold_change',p_adj_threshold=.05,nan_lfc_policy='mask',min_abs_log2fc=0)
    tidy_rows=dispatch_de_metrics(metric_names,prepared,score_cfg)
    sc=exact_mse(P,J,verify=True)
    pds=discrimination_score(pred_bulk=(np.array(targets),P),real_bulk=(perts,means),distance='cosine',
       rank_denominator='n-1',tie_policy='midrank',exclude_target_gene=True,exclusion_scope='panel',
       control_source='real',genes=genes,control='non-targeting')
    for i,t in enumerate(targets):
        for metric,value in [('expr_mse_unbiased',sc['uncapped'][i]),('expr_mse_unbiased_capped',sc['capped'][i]),
                             ('expr_distance_unbiased',Dden[i]),('pds_cosine',pds[t])]:
            tidy_rows.append(dict(perturbation=t,metric=metric,value=float(value)))
    agg=aggregate_metrics(pl.DataFrame(tidy_rows));raw=dict(zip(agg['metric'].to_list(),agg['mean'].to_list()))
    out={short_names[m]:float(score_one(raw[m],entries[m].base,entries[m].scoring)) for m in metric_names}
    out.update(overall=float(np.mean(list(out.values()))),mse_unclamped=scaled_mse(sc['raw']),mse_raw=sc['raw'],
       mse_raw_uncapped=sc['raw_uncapped'],correction_budget_ratio=sc['ratio'])
    return out,pl.DataFrame(tidy_rows)
# Prove all six members and their overall reproduce the existing scorer run from cached DE.
for name in ['arm1','arm2','arm7']:
    got,_=six_scores(regenerated[name]['P'],regenerated[name]['J'],pl.read_parquet(arm_paths[name]/'de_pred.parquet'))
    old=pd.read_csv(arm_paths[name]/'scaled.csv')
    # The saved summary is a simpler exact reference for the same six scalings.
    summary=pd.read_csv(arm_paths[name].parent/'scaled_scores.csv',index_col=0).loc[arm_paths[name].name]
    for m,short in short_names.items():np.testing.assert_allclose(got[short],summary[m],atol=3e-5,rtol=3e-5)
    np.testing.assert_allclose(got['overall'],summary['overall'],atol=1e-5)
    cell_results.append(dict(arm=name,**got))

def emit_cells(Dc,i,seed=2026,method='original',pick=None):
    local=copy.copy(cfg);local.seed=seed
    if pick is None:
        pc=copy.copy(pool_cfg)
        base_seed=seed_for(seed,'H1','base_controls') if method!='independent' else seed_for(seed,'H1',targets[i],'base_controls')
        pick=gen.pick_cells(line,np.zeros(G),pc,np.random.default_rng(base_seed))[:400]
    rng=np.random.default_rng(seed_for(seed,'research_mse',targets[i],method))
    F,proj,_=gen.project_factors(line.share,Dc,int(own[i]),kd[i])
    if method=='round':
        expected=line.X[pick].toarray().astype(float)*F[None,:]
        # Zero-baseline induction follows the existing projection convention.
        zero=(line.share==0)&(proj>0)
        expected[:,zero]=line.lib[pick,None]*proj[None,zero]
        values=np.floor(expected).astype(np.int64)+rng.binomial(1,expected-np.floor(expected))
        X=sp.csr_matrix(values.astype(np.int32))
    else:
        X=gen.generate_one(line,Dc,int(own[i]),kd[i],local,pick=pick,line_key='H1',target_key=targets[i])[0]
    if method=='full_totals':
        target_totals=400*full_mean*F
        target_totals[(line.share==0)&(proj>0)]=400*full_mean.sum()*proj[(line.share==0)&(proj>0)]
        X=sp.csr_matrix(pin_totals(X.toarray(),target_totals,rng).astype(np.int32))
        assert np.array_equal(np.asarray(X.sum(0)).ravel(),np.rint(target_totals).astype(np.int64))
    gen.validate_counts(X)
    assert np.asarray(X.sum(1)).max()<=1e6
    return X

def evaluate_cells(name,Dc,method='original',seed=2026,pick_fn=None):
    prefix=OUT/name;meta=prefix.with_suffix('.json');moment=prefix.with_suffix('.npz');depath=prefix.with_suffix('.parquet')
    if meta.exists() and moment.exists() and depath.exists():
        row=json.loads(meta.read_text());cell_results.append(row);return row
    P=np.empty((T,G));J=np.empty(T);de_frames=[];start=time.time()
    for i,t in enumerate(targets):
        X=emit_cells(Dc[i],i,seed,method,None if pick_fn is None else pick_fn(i,seed))
        sums=np.asarray(X.sum(0)).ravel().astype(float);P[i]=np.log1p(50000*sums/sums.sum())
        J[i]=jackknife_correction(X,np.zeros(400,np.int32),1,50000,chunk=32)[0]
        de_frames.append(predicted_de(X,t))
        if (i+1)%5==0:progress(f'{name}: {i+1}/{T}, {time.time()-start:.1f} s')
    table=pl.concat(de_frames);table.write_parquet(depath);np.savez_compressed(moment,P=P,J=J)
    result,tidy_result=six_scores(P,J,table);row=dict(arm=name,method=method,seed=seed,**result)
    tidy_result.write_csv(OUT/f'{name}_metrics.csv');meta.write_text(json.dumps(row,indent=2));cell_results.append(row)
    pd.DataFrame(cell_results).drop_duplicates('arm',keep='last').to_csv(OUT/'phase3_cell_results.csv',index=False)
    progress(row);return row


In [ ]:
# Exact repeated-reference MWU: cache sorted controls, then binary-search 400 target values.
# The U, zero ties, other ties, continuity correction, and normal CDF match numba_mwu.
# This avoids re-sorting ~19,000 controls for every target and every arm.
from numba_mwu._core import _ndtr
from numba import njit,prange
import math
@njit(parallel=True)
def prepare_sorted_reference(data,order,ptr,n):
    values=np.empty(len(order),np.float64);ties=np.zeros(len(ptr)-1)
    for g in prange(len(ptr)-1):
        start,end=ptr[g],ptr[g+1];a=np.sort(data[order[start:end]])
        values[start:end]=a;j=0;ts=0.
        while j<len(a):
            k=j+1
            while k<len(a) and a[k]==a[j]:k+=1
            t=float(k-j);ts+=t*t*t-t;j=k
        ties[g]=ts
    return values,ties
rvals,rties=prepare_sorted_reference(ref_index.data,ref_index.col_order,ref_index.col_indptr,control_X.shape[0])
rptr=ref_index.col_indptr;nref=control_X.shape[0]
@njit(parallel=True)
def reference_mwu(xdata,xptr,na,rvals,rptr,nt,ref_ties):
    out=np.empty(len(xptr)-1)
    for g in prange(len(out)):
        x=np.sort(xdata[xptr[g]:xptr[g+1]]);y=rvals[rptr[g]:rptr[g+1]]
        za=na-len(x);zb=nt-len(y);z=float(za+zb)
        tie=z*z*z-z+ref_ties[g];U=float(za)*float(zb)/2
        j=0
        while j<len(x):
            k=j+1
            while k<len(x) and x[k]==x[j]:k+=1
            lo=np.searchsorted(y,x[j],'left');hi=np.searchsorted(y,x[j],'right')
            aa=float(k-j);bb=float(hi-lo);tot=aa+bb
            U+=aa*(zb+lo+bb/2)
            tie+=tot*tot*tot-tot-(bb*bb*bb-bb)
            j=k
        N=na+nt;variance=na*nt/12.*((N+1.)-tie/(float(N)*float(N-1)))
        if variance<=0:out[g]=1.
        else:
            zscore=(abs(U-na*nt/2.)-.5)/math.sqrt(variance)
            out[g]=min(1.,max(0.,2*_ndtr(-zscore)))
    return out

def fast_pvalues(logX):
    c=logX.tocsc();c.eliminate_zeros()
    return reference_mwu(c.data.astype(np.float64),c.indptr,c.shape[0],rvals,rptr,nref,rties)
# Validate against the official pdex kernel, including a second seed and strong mixed signs.
for test_d,test_seed in [(D2[0],2026),(D7[0],2027)]:
    xx=emit_cells(test_d,0,test_seed)
    ll=_log1p_view(_to_linear(ad.AnnData(xx),'counts',1e6)).X[:,de_gate].tocsr()
    slow=np.clip(np.asarray(mwu(ll,ref_index).pvalue),0,1);fast=fast_pvalues(ll)
    np.testing.assert_allclose(fast,slow,rtol=2e-12,atol=1e-14)
    print('Cached-sorted MWU exact validation max p difference:',float(np.max(np.abs(fast-slow))),flush=True)
def predicted_de(X,target):
    lin=_to_linear(ad.AnnData(X,var=pd.DataFrame(index=genes)),'counts',1e6)
    mean=_grouped_means(lin.X,np.arange(X.shape[0]),np.array([0,X.shape[0]]),1,log_space=False)[0]
    p=fast_pvalues(_log1p_view(lin).X[:,de_gate].tocsr())
    lfc=_lfc_from_means({target:mean[de_gate],'non-targeting':ref_mean[de_gate]},genes[de_gate],reference='non-targeting',epsilon=1e-9)
    return _bh_per_target(lfc.with_columns(pl.Series('p_value',p),pl.Series('p_adj',np.ones(len(p)))))


In [ ]:
# E1a: preserve stochastic cells, correct totals to the entire generator-control pool.
evaluate_cells('E1a_arm7_fullcontrols',D7,method='full_totals')
evaluate_cells('E1a_arm2_fullcontrols',D2,method='full_totals')
display(pd.DataFrame(cell_results).drop_duplicates('arm',keep='last'))


**E5 validation export provenance.** The following isolated script was launched in `research_mse_valkd` tmux on vcc2027. It performs inference only, uses every training-line `val_kd` row, never loads H1 effect values, and writes new research artifacts. Batch 8 and two CPU threads keep inference resource use modest. Existing jobs are untouched.

In [ ]:
VALIDATION_EXPORT_SOURCE = '"""Isolated export; source is embedded in research_mse.ipynb."""\nfrom pathlib import Path\nimport sys,time,json\nimport numpy as np\nimport torch\nV=Path(\'/home/mooneysad/vcc\');sys.path.insert(0,str(V/\'code\'))\nfrom transformer.evaluate import load_run\nfrom transformer.data import LINES\nout=V/\'results/research_mse/val_kd_export\';out.mkdir(parents=True,exist_ok=True)\ntorch.set_num_threads(2)\ncfg,bank,model=load_run(V/\'lib/vcc_2026_4/runs/dev_nopie\',device=\'cuda\')\nassert bank.mode==\'validation\' and \'H1\' not in bank.loaded\nrows=bank.val_rows;bs=8\nparts={k:[] for k in [\'pred\',\'true\',\'mask\',\'coverage\']};st=time.time()\nwith torch.inference_mode():\n for s in range(0,len(rows),bs):\n  b=bank.batch(rows=rows[s:s+bs],train=False)\n  with torch.autocast(\'cuda\',dtype=torch.bfloat16):p=model(b)[\'delta\']\n  for k,v in [(\'pred\',p.float()),(\'true\',b[\'lfc_t\']),(\'mask\',b[\'loss_mask\']),(\'coverage\',b[\'slot_mask\'].sum(-1))]:\n   dt=np.bool_ if k==\'mask\' else np.uint8 if k==\'coverage\' else np.float32\n   parts[k].append(v.cpu().numpy().astype(dt))\n  if s%128==0: print(f\'{s}/{len(rows)} {time.time()-st:.1f}s\',flush=True)\nnp.savez_compressed(out/\'validation.npz\',**{k:np.concatenate(v) for k,v in parts.items()},line=bank.row_line[rows],target=bank.kd_symbol[bank.row_kd[rows]].astype(str),gcols=bank.gcols,basal=bank.basal.cpu().numpy(),line_names=np.array(LINES))\n(out/\'metadata.json\').write_text(json.dumps(dict(rows=len(rows),config=cfg,loaded=bank.loaded,elapsed=time.time()-st),indent=2))\nprint(\'COMPLETE\',time.time()-st,flush=True)\n'
print(VALIDATION_EXPORT_SOURCE)

### Completed Phase 3 runs — consolidated 21 October 2026

The following cells retain outputs from two isolated worker kernels. Their setup was identical to the setup above; one worker queued E5/E6 cell evaluations for the other worker. The consolidated notebook calls the ordinary cached evaluator directly. Execution counters reflect those separate kernels, not a single chronological run. All 14 saved cell experiments completed without notebook execution errors. The cached-sorted MWU implementation above matched the official `pdex` kernel on two H1 examples; additional sparse, dense, continuous, all-zero and constant synthetic examples passed the same tolerance (`rtol=2e-12`, `atol=1e-14`).

Oracle diagnostics are diagnostic only. They use H1 truth and cannot be candidate predictions. The raw oracle calls below do not correct the covariance from estimating the shared profile on the same targets; the earlier E3c calculation gives the explicitly covariance-adjusted approximation.

In [ ]:
evaluate_cells('arm7_current_backend',D7)

**E1b/c: representative base cells and common random numbers.** The original generator already shares its base pool across knockdowns (E1c). We compare an independent pool to that reference, and a guide/library-size-stratified greedy herding design. Herding minimizes a fixed 64-dimensional random projection of the full-panel bulk-log derivative, plus library size, using each stratum in proportion to its population. It selects distinct cells within a knockdown and uses target-specific seeds. This is approximate full-panel herding; the actual pseudobulk error is measured independently. Seed diagnostics use ten evenly spaced targets and seeds 2026/2027; aggregate six-metric tests always use all 75 targets.

In [ ]:
from sklearn.random_projection import SparseRandomProjection
herd_cache=OUT/'herding_embedding.npz'
if herd_cache.exists():
    hz=np.load(herd_cache);Zherd=hz['Z'];stratum=hz['stratum']
else:
    derivative=50000/(1+50000*line.share.astype(float))/full_mean.sum()
    projector=SparseRandomProjection(n_components=64,density=.02,random_state=2026)
    Zherd=projector.fit_transform(line.X.multiply(derivative).tocsr())
    Zherd=Zherd.toarray() if sp.issparse(Zherd) else Zherd
    Zherd=np.column_stack([Zherd,line.lib/line.lib.mean()*.1]).astype(np.float64)
    Zherd-=Zherd.mean(0)
    stratum=np.zeros(line.n,int)
    for j,group in enumerate(line.groups):
        quant=pd.qcut(pd.Series(line.lib[group]).rank(method='first'),10,labels=False).to_numpy()
        stratum[group]=j*10+quant
    np.savez_compressed(herd_cache,Z=Zherd,stratum=stratum)
stratum_groups={s:np.flatnonzero(stratum==s) for s in np.unique(stratum)}
def herded_pick(i,seed):
    path=OUT/f'herd_{seed}_{targets[i]}.npy'
    if path.exists():return np.load(path)
    rng=np.random.default_rng(seed_for(seed,'herd',targets[i]))
    groups=list(stratum_groups);sizes=np.array([len(stratum_groups[s]) for s in groups])
    # Systematic stratified sampling preserves proportional stratum counts to within one cell.
    u=(np.arange(400)+rng.random())/400
    schedule=np.array(groups)[np.searchsorted(np.cumsum(sizes)/sizes.sum(),u)]
    rng.shuffle(schedule);residual=np.zeros(Zherd.shape[1]);chosen=[];used=np.zeros(line.n,bool)
    for s in schedule:
        opts=stratum_groups[s];opts=opts[~used[opts]]
        if len(opts)>128:opts=rng.choice(opts,128,replace=False)
        cost=2*(Zherd[opts]@residual)+(Zherd[opts]**2).sum(1)
        j=int(opts[np.argmin(cost)]);chosen.append(j);used[j]=True;residual+=Zherd[j]
    chosen=np.array(chosen);np.save(path,chosen);return chosen
noise_path=OUT/'phase3_seed_noise.csv'
if noise_path.exists():noise_report=pd.read_csv(noise_path)
else:
    rec=[];subset=np.linspace(0,T-1,10,dtype=int)
    for method in ['original','independent','herding','full_totals','round']:
        Ps=[]
        for seed in [2026,2027]:
            P=[]
            for i in subset:
                X=emit_cells(D7[i],i,seed,'original' if method=='herding' else method,
                             herded_pick(i,seed) if method=='herding' else None)
                sm=np.asarray(X.sum(0)).ravel().astype(float);P.append(np.log1p(50000*sm/sm.sum()))
            Ps.append(np.array(P))
        m=mask[subset];half=.5*np.mean((Ps[0]-Ps[1])[m]**2)
        rec.append(dict(method=method,half_mse=half,half_mse_over_mean_normalizer=half/Dden.mean(),
            rms_from_intended=np.sqrt(np.mean((Ps[0]-P7[subset])[m]**2))))
        progress(rec[-1])
    noise_report=pd.DataFrame(rec);noise_report.to_csv(noise_path,index=False)
display(noise_report)
evaluate_cells('E1b_arm7_herding',D7,pick_fn=herded_pick)
evaluate_cells('E1c_arm7_independent',D7,method='independent')


**E2: floor constraints and simulated proportional scaling.** The one-change floor-cap/CPM variants keep all other arm-7 settings. For E2c, 400 sampled controls are multiplied by composition-projected factors and stochastically rounded to integer counts. A grid of 0.03–1 log2FC is simulated in both directions, using two control-sampling seeds and complementary sign templates on the model genes. A smaller floor is accepted only when both simulations pass the scorer's control CPM gate, Wilcoxon and per-target BH 0.05 in the intended direction. Unreached or larger thresholds retain the old floor. The simulation uses controls and model outputs only, never H1 perturbation truth. A round-with-old-floor arm isolates the generator effect from the floor recalibration.

In [ ]:
evaluate_cells('E2a_cap0p5',candidates['E2a_skip_above_0.5']['D'])
evaluate_cells('E2b_cpm',candidates['E2b_cpm_thresholds']['D'])
evaluate_cells('E2c_round_oldfloor',D7,method='round')
floor_sim_path=OUT/'proportional_simulated_floor.npz'
if floor_sim_path.exists():
    sf=np.load(floor_sim_path);sim_up=sf['up'];sim_down=sf['down']
else:
    mag_grid=[.03,.06,.1,.15,.2,.3,.4,.5,.75,1.0]
    direction=np.where(delta.mean(0)>=0,1.,-1.)
    # A template and its complement cover up/down for every modeled gene.
    sim_up=np.full(G,np.inf);sim_down=np.full(G,np.inf)
    for mag in mag_grid:
        for sign in [1,-1]:
            dc=np.zeros(G);dc[gcols]=mag*direction*sign;dc[~line.gate]=0;dc[own[0]]=0
            hits=[]
            for seed in [3026,3027]:
                X=emit_cells(dc,0,seed,method='round')
                de=predicted_de(X,targets[0]).to_pandas().set_index('feature').reindex(genes)
                hits.append((de.p_adj.to_numpy()<.05)&(np.sign(de.log2_fold_change.to_numpy())==np.sign(dc)))
            both=hits[0]&hits[1]
            sim_up[(dc>0)&both&~np.isfinite(sim_up)]=mag
            sim_down[(dc<0)&both&~np.isfinite(sim_down)]=mag
        progress(f'floor simulation magnitude {mag}: up {np.isfinite(sim_up).sum()}, down {np.isfinite(sim_down).sum()}')
    np.savez_compressed(floor_sim_path,up=sim_up,down=sim_down)
new_up=np.minimum(up,sim_up);new_down=np.minimum(down,sim_down)
Dsim=D7_no.copy();req=np.where(Dsim>0,new_up[None,:],new_down[None,:])
Dsim=np.where(Dsim!=0,np.sign(Dsim)*np.maximum(np.abs(Dsim),req),0)
Dsim[:,~line.gate]=0;Dsim[np.arange(T),own]=0
screen_arm('E2c_simulated_round_floor',Dsim,'E2c proportional')
report=pd.DataFrame({'gene':genes,'old_up':up,'new_up':new_up,'old_down':down,'new_down':new_down,'sim_up':sim_up,'sim_down':sim_down})
report.to_csv(OUT/'phase3_simulated_floor.csv',index=False)
print('Floors reduced:',int((new_up<up).sum()),int((new_down<down).sum()))
display(report.loc[run_set & line.gate,['old_up','new_up','old_down','new_down']].describe())
evaluate_cells('E2c_simulated_round_floor',Dsim,method='round')


**E3 and E4.** Shared β and source extension are screened individually. The extension has a large pseudobulk penalty, but one representative cell arm checks whether DE gains offset it. The oracle shared profile is reported at both s=1 and s=2 and is never deployable. The global per-target offset accounts for <1% of arm-7 plugin error; an oracle removal sizes its maximum benefit. Multiplying every count in a cell by a common scalar cancels in this scorer's bulk composition, so it is not a deployable correction to this residual.

In [ ]:
evaluate_cells('E3a_beta2',candidates['E3a_beta2_floor']['D'])
evaluate_cells('E3b_extension',candidates['E3b_beta1_extension_floor']['D'])
for name,P in [('s1',P2),('s2',P7_no)]:
    oracle=R.mean(0)+(P-P.mean(0))
    q=exact_mse(oracle)
    print('Oracle shared',name,'raw MSE',q['raw'],'unclamped',scaled_mse(q['raw']))
for name,P in [('arm2',regenerated['arm2']['P']),('arm7',regenerated['arm7']['P'])]:
    correction=((P-R)*mask).sum(1)/nrow
    corrected=P-correction[:,None]
    q=exact_mse(corrected,regenerated[name]['J'])
    print('E4 oracle constant-offset removal',name,'raw',q['raw'],'unclamped',scaled_mse(q['raw']))
pd.DataFrame(screen).to_csv(OUT/'phase3_pseudobulk_screen.csv',index=False)
display(pd.DataFrame(cell_results).drop_duplicates('arm',keep='last').sort_values('overall',ascending=False))


**E5: training-only reliability fit.** Every `val_kd` row from the six training destinations is exported from the frozen dev checkpoint. H1 is absent from targets and evidence. Within each line/gene, predictions and truths are centered over measured validation knockdowns. Nonnegative least-squares slopes, clipped to [0,1], shrink the specific component by control-expression rank quintile × evidence-coverage bucket (0, 1, 2, ≥3). Each destination has equal total weight; a 200-entry-equivalent prior toward the pooled slope stabilizes small buckets. This is an LFC-space reliability proxy, not an exact fit to the single-cell MSE objective, since training-line cell-level jackknives are unavailable. H1 is used only to evaluate the frozen fit. We compare the requested global-size grid and cell-score the best reliability variant.

In [ ]:
import h5py
from anndata.io import read_elem
sys.path.insert(0,str(V/'code'))
from transformer.data import basal_table
vz=np.load(OUT/'val_kd_export/validation.npz');val_pred=vz['pred'];val_true=vz['true'];val_mask=vz['mask'];val_cov=vz['coverage'];val_line=vz['line']
assert np.array_equal(vz['gcols'],gcols)
val_names=vz['line_names'];assert 'H1' not in set(val_names[np.unique(val_line)])
Xc=np.zeros_like(val_pred);Yc=np.zeros_like(val_true);W=np.zeros_like(val_pred)
for li in np.unique(val_line):
    keep=val_line==li;m=val_mask[keep];den=m.sum(0).clip(1)
    Xc[keep]=val_pred[keep]-(val_pred[keep]*m).sum(0)/den
    Yc[keep]=val_true[keep]-(val_true[keep]*m).sum(0)/den
    W[keep]=m/m.sum().clip(1)
base_slope=float(np.sum(W*Xc*Yc)/np.sum(W*Xc*Xc));print('Training-validation pooled slope',base_slope)
# basal channels: detection, missing, expression percentile, log CPM, measured.
quint=np.clip((vz['basal'][val_line][:,gcols,2]*5).astype(int),0,4)
coverage_bucket=np.minimum(val_cov,3);bucket=quint*4+coverage_bucket
slopes=np.full(20,np.clip(base_slope,0,1));records=[]
prior=200*np.sum(W*Xc*Xc)/val_mask.sum()
for k in range(20):
    sel=(bucket==k)&val_mask;xy=np.sum((W*Xc*Yc)[sel]);xx=np.sum((W*Xc*Xc)[sel])
    slopes[k]=np.clip((xy+prior*np.clip(base_slope,0,1))/(xx+prior),0,1)
    records.append(dict(expression_quintile=k//4,coverage_bucket=k%4,n=int(sel.sum()),slope=slopes[k]))
reliability=pd.DataFrame(records);reliability.to_csv(OUT/'phase3_reliability.csv',index=False);display(reliability)
# H1 coverage is derived exclusively from non-H1 sources.
from v5lib import SOURCE_FILES
h1coverage=np.zeros((len(t300),len(gcols)),np.uint8)
for src in source_names:
    with h5py.File(V/'lib/effects_v4'/f'{SOURCE_FILES[src]}.h5ad') as f:
        obs=read_elem(f['obs']);pos={str(t):j for j,t in enumerate(obs.target_gene)}
        for i,t in enumerate(t300):
            if t in pos:
                l=f['layers/lfc'][pos[t]][gcols];se=f['layers/se'][pos[t]][gcols]
                h1coverage[i]+=np.isfinite(l)&np.isfinite(se)
bh,_=basal_table(V/'lib/effects_v4','H1',genes)
h1quint=np.clip((bh[gcols,2]*5).astype(int),0,4)
h1s=slopes[h1quint[None,:]*4+np.minimum(h1coverage,3)]
shrunk=delta.mean(0)+(delta-delta.mean(0))*h1s
Ds=np.zeros((T,G));Ds[:,gcols]=shrunk[idx]
for floor in [False,True]:
    _,Dc,_=intended(D=Ds,floor=floor)
    screen_arm(f'E5_reliability_{"floor" if floor else "nofloor"}',Dc,'E5 training reliability')
np.savez_compressed(OUT/'phase3_reliability_fit.npz',slopes=slopes,h1_factors=h1s,pooled_slope=base_slope)
# Two informative cell comparisons: no floor for MSE, existing floor for overall.
evaluate_cells('E5_reliability_nofloor',candidates['E5_reliability_nofloor']['D'])
evaluate_cells('E5_reliability_floor',candidates['E5_reliability_floor']['D'])


**E6: low-rank and amplitude screens.** PCA is fitted only to the six training lines, over model genes measured in ≥4 of them. Effects are centered within line; unmeasured entries are mean-imputed (zero after centering), target-gene entries are excluded, and each line contributes total weight one. Randomized SVD computes 50 components. Structure summaries use the same gene set in each line and H1; they are descriptive observed-effect variance, without measurement-noise subtraction. The amplitude ridge uses [leave-destination-out mean c, DepMap mean dependency across its 1,186 lines, median source on-target LFC, mean source effect norm]. Missing covariates are median-imputed within training folds. Target genes are grouped across destinations for model-selection CV, preventing the same knockdown appearing on both sides of a fold. All sources are non-H1 for H1 prediction; the source summaries never use the destination's effects. The ridge's training labels and source summaries are in effect-table log2FC space; the final amplitude replacement is also made in that space before composition projection.

In [ ]:
from sklearn.utils.extmath import randomized_svd
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from scipy.stats import spearmanr
train_lines=['K562','RPE1','HepG2','Jurkat','HCT116','HEK293T']
training={};obs_by_line={};coverage=[]
for ln in train_lines+['H1']:
    with h5py.File(V/'lib/effects_v4'/f'{SOURCE_FILES[ln]}.h5ad') as f:
        obs=read_elem(f['obs']);L=np.empty((len(obs),len(gcols)),np.float32)
        for start in range(0,len(obs),512):L[start:start+512]=f['layers/lfc'][start:start+512][:,gcols]
    colmap={g:j for j,g in enumerate(genes[gcols])}
    for j,t in enumerate(obs.target_gene.astype(str)):
        if t in colmap:L[j,colmap[t]]=np.nan
    training[ln]=L;obs_by_line[ln]=obs
    if ln!='H1':coverage.append(np.isfinite(L).sum(0)>=30)
common_gene=np.array(coverage).sum(0)>=4
print('E6 genes measured in at least four training lines:',int(common_gene.sum()))
train_matrices=[];generics={};amplitude={};strengths={};structure=[]
for ln,L in training.items():
    A=L[:,common_gene];generic=np.nan_to_num(np.nanmean(A,axis=0));centered=np.nan_to_num(A-generic).astype(np.float32)
    generics[ln]=generic
    amplitude[ln]=centered@generic/max(float(generic@generic),1e-12)
    strengths[ln]=np.sqrt(np.nanmean(A*A,axis=1))
    if ln!='H1':train_matrices.append(centered/np.sqrt(len(A)))
    U,s,Vt=randomized_svd(centered,n_components=min(30,min(centered.shape)-1),n_iter=3,random_state=2026)
    total=float(np.sum(centered.astype(float)**2));specific=amplitude[ln][:,None]*generic
    record={'line':ln,'n':len(A),'amplitude_fraction':float(np.sum(specific.astype(float)**2)/total)}
    record.update({f'pc{k}':float(np.sum(s[:k]**2)/total) for k in [1,3,10,30]})
    structure.append(record);progress(record)
structure=pd.DataFrame(structure);structure.to_csv(OUT/'phase3_structure.csv',index=False);display(structure)
fitmat=np.concatenate(train_matrices);_,singular,pcs=randomized_svd(fitmat,n_components=50,n_iter=5,random_state=2026);del fitmat,train_matrices
np.savez_compressed(OUT/'training_pca.npz',components=pcs,genes=genes[gcols][common_gene],singular=singular)
specific=delta-delta.mean(0);sub=specific[:,common_gene]
for K in [5,10,20,50]:
    projection=(sub@pcs[:K].T)@pcs[:K]
    for lam in [.25,.5,.75,1.]:
        d=specific.copy();d[:,common_gene]+=lam*(projection-sub)
        D=np.zeros((T,G));D[:,gcols]=(delta.mean(0)+d)[idx]
        _,Dc,_=intended(D=D)
        screen_arm(f'E6a_K{K}_lambda{lam:g}',Dc,'E6a PCA')
# c and strength lookup tables have one observation per source line/target.
lookup={}
for ln in train_lines:
    for j,t in enumerate(obs_by_line[ln].target_gene.astype(str)):
        lookup[(ln,t)]=(float(amplitude[ln][j]),float(obs_by_line[ln].efficiency.iloc[j]),float(strengths[ln][j]))
dep=np.load(V/'lib/pie_features/depmap_gene_effect.npy',mmap_mode='r')
depmean=np.nanmean(dep.astype(np.float32),axis=1);assert len(depmean)==G
def amplitude_features(t,dest):
    src=np.array([lookup[(ln,t)] for ln in train_lines if ln!=dest and (ln,t) in lookup],float)
    source=np.nanmean(src,axis=0) if len(src) else np.full(3,np.nan)
    med=np.nanmedian(src[:,1]) if len(src) and np.isfinite(src[:,1]).any() else np.nan
    return [source[0],float(depmean[gp[t]]) if t in gp else np.nan,med,source[2]]
features=[];labels=[];groups=[];line_ids=[]
for ln in train_lines:
    for j,t in enumerate(obs_by_line[ln].target_gene.astype(str)):
        if any((other,t) in lookup for other in train_lines if other!=ln):
            features.append(amplitude_features(t,ln));labels.append(amplitude[ln][j]);groups.append(t);line_ids.append(ln)
features=np.array(features);labels=np.array(labels);groups=np.array(groups);line_ids=np.array(line_ids)
line_counts={ln:int((line_ids==ln).sum()) for ln in train_lines}
sample_weight=np.array([1/line_counts[ln] for ln in line_ids]);sample_weight*=len(labels)/sample_weight.sum()
cv=GroupKFold(5);fit_scores=[]
for alpha in [1,10,100,1000]:
    oof=np.empty(len(labels))
    for tr,te in cv.split(features,labels,groups):
        model=make_pipeline(SimpleImputer(add_indicator=True),StandardScaler(),Ridge(alpha=alpha))
        model.fit(features[tr],labels[tr],ridge__sample_weight=sample_weight[tr]);oof[te]=model.predict(features[te])
    fit_scores.append(dict(alpha=alpha,mse=float(np.average((oof-labels)**2,weights=sample_weight)),spearman=float(spearmanr(oof,labels).statistic)))
fit_scores=pd.DataFrame(fit_scores);display(fit_scores);bestalpha=float(fit_scores.sort_values('mse').iloc[0].alpha)
model=make_pipeline(SimpleImputer(add_indicator=True),StandardScaler(),Ridge(alpha=bestalpha))
model.fit(features,labels,ridge__sample_weight=sample_weight)
chat=model.predict(np.array([amplitude_features(t,'H1') for t in t300]))
hpos={t:j for j,t in enumerate(obs_by_line['H1'].target_gene.astype(str))}
htrue=np.array([amplitude['H1'][hpos[t]] for t in t300]);amp_r=float(spearmanr(chat,htrue).statistic)
print('E6b H1 Spearman(c_hat,c):',amp_r,'ridge alpha',bestalpha)
Rhat=delta.mean(0);dhat=delta-Rhat;old_amplitude=dhat@Rhat/max(float(Rhat@Rhat),1e-12)
new=Rhat+dhat-old_amplitude[:,None]*Rhat+(chat-chat.mean())[:,None]*Rhat
D=np.zeros((T,G));D[:,gcols]=new[idx];_,Dc,_=intended(D=D)
screen_arm('E6b_amplitude_ridge',Dc,'E6b amplitude')
pd.DataFrame({'target':t300,'c_hat':chat,'c_real':htrue}).to_csv(OUT/'phase3_amplitude_h1.csv',index=False)
# E6 is explicitly a quick pseudobulk screen. Cell-test a promising winner only.
e6=pd.DataFrame(screen).query("family.str.startswith('E6')",engine='python').sort_values('mse_raw')
display(e6)
if e6.iloc[0].mse_raw < exact_mse(P2)['raw']:
    winner=e6.iloc[0].arm;evaluate_cells(winner,candidates[winner]['D'])
fig,axs=plt.subplots(1,2,figsize=(12,4))
structure.set_index('line')[[f'pc{k}' for k in [1,3,10,30]]].plot.bar(ax=axs[0]);axs[0].set_ylabel('Observed specific variance explained')
axs[1].scatter(chat,htrue,s=10,alpha=.6);axs[1].set(xlabel='Training-only predicted amplitude',ylabel='H1 observed amplitude',title=f'Spearman r={amp_r:.3f}')
fig.tight_layout();fig.savefig(OUT/'phase3_structure_amplitude.png',dpi=160);plt.show()
pd.DataFrame(screen).to_csv(OUT/'phase3_pseudobulk_screen.csv',index=False)


In [ ]:
# Full six-metric outputs from the cell worker; cached on a consolidated rerun.
for name in ['E5_reliability_nofloor', 'E5_reliability_floor', 'E6b_amplitude_ridge']:
    evaluate_cells(name, candidates[name]['D'])


### Research closeout

The scorer audit, error decomposition, and completed Phase 3 screens/cell experiments are consolidated here. Phase 4 A–C candidate generation and packaging were **not completed**: the user redirected the task to reviewing the evidence and public competition methods. No submission was uploaded. These are **local H1 results**, not official A–C leaderboard scores.

The full-control-total correction removes the sampled-control shift while retaining integer cells. Without the detection floor it yields raw MSE **0.931684**, below the **0.953469** baseline, hence **+0.022902 scaled MSE**. Positive H1 MSE is therefore demonstrated, not a 30% possibility. Its overall score is only **0.049511**. Keeping the floor and applying the same totals correction gives **0.169113 overall**, versus **0.164225** for the current floor-arm evaluator; both have zero scaled MSE. Other completed cell arms did not improve on that overall result.

The two-seed, ten-target diagnostic shows substantial *random* sampled-control error relative to the small MSE denominator. Because a common control sample is reused across perturbations, one realization looks like a shared-response bias. This is not proof of irreducible deterministic bias. Full-pool totals remove that seed variation; residual RMS from intended bulk is about 0.000391.

Training-only reliability calibration and the low-rank/amplitude screens did not provide the needed joint improvement. The amplitude ridge has H1 Spearman correlation about 0.211, but its generated cells still score poorly. Simply extending the shared response outside the model genes was harmful. All-gene *target-specific* modeling remains a distinct, untested proposal.

The attached error-budget explanation is useful as a local decomposition, but its four conditions are not mathematical necessities. The scorer's context baseline is a constructed count-space baseline, not an exactly known mean log response, and 0.05 is the no-change-versus-baseline gap on this H1 panel—not a universal ceiling on attainable improvement. The proposed sign factor `2p-1` requires a calibrated sign probability under an explicit magnitude model. Our transformer head predicts **DE membership**, not sign; substituting it would reverse effects below 0.5.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path('/home/mooneysad/vcc')
RES = ROOT/'results/research_mse'
rows = []
for path in sorted(RES.glob('*.json')):
    item = json.loads(path.read_text())
    if isinstance(item, dict) and {'arm','overall','mse_raw','mse'} <= item.keys():
        rows.append(item)
completed = pd.DataFrame(rows).sort_values('overall', ascending=False)
assert len(completed) == 14
assert np.allclose(completed[['pds','mse','nmae','fid','reach','jac']].mean(axis=1), completed.overall)
completed.to_csv(RES/'phase3_cell_results_consolidated.csv', index=False)
display(completed[['arm','overall','mse_raw','mse','pds','nmae','fid','reach','jac']].round(6))
display(pd.read_csv(RES/'phase3_seed_noise.csv'))
fig, ax = plt.subplots(figsize=(9,6))
ax.scatter(completed.mse_raw, completed.overall, s=45)
for name, label in [('arm7_current_backend','Current floor arm'),
                    ('E1a_arm7_fullcontrols','Pinned totals + floor'),
                    ('E1a_arm2_fullcontrols','Pinned totals, no floor')]:
    row = completed.set_index('arm').loc[name]
    ax.annotate(label, (row.mse_raw,row.overall), xytext=(5,8), textcoords='offset points', fontsize=9)
ax.axvline(.9534691116197181, color='black', linestyle='--', label='H1 MSE baseline')
ax.set(xlabel='Raw MSE (lower is better)', ylabel='Overall scaled score', title='H1: positive MSE and best overall remain different arms')
ax.legend(); fig.tight_layout(); fig.savefig(RES/'phase3_completed_tradeoff.png', dpi=160); plt.show()

# Read target identifiers only; no raw-cell processing or fitting.
import h5py
from anndata.io import read_elem
source_files = {'HepG2':'nadig_hepg2','K562_GWPS':'replogle_k562_gw',
 'CD4Stim48':'marson_cd4_stim48hr','RPE1':'replogle_rpe1','H1':'arc_h1',
 'HCT116':'orion_hct116','CD4Stim8':'marson_cd4_stim8hr','Jurkat':'nadig_jurkat',
 'HEK293T':'orion_hek293t','CD4Rest':'marson_cd4_rest'}
target_sets = {}
for name, stem in source_files.items():
    with h5py.File(ROOT/'lib/effects_v4'/f'{stem}.h5ad') as f:
        target_sets[name] = set(read_elem(f['obs']).target_gene.astype(str))
abc = set(pd.read_csv(ROOT/'data/pert_counts.csv').target_gene.astype(str))
h1 = target_sets['H1']
coverage = pd.DataFrame([dict(source=name, ABC_targets=len(s & abc), H1_targets=len(s & h1))
                         for name,s in target_sets.items()])
coverage.to_csv(RES/'source_target_coverage.csv', index=False)
print('Panel sizes: ABC',len(abc),'H1',len(h1))
display(coverage)


### Public-method review and priority changes — 21 October 2026

The [official validation leaderboard API](https://virtualcellchallenge.org/api/leaderboard?get_final=false) returned 1,621 entries with 1,621 distinct team IDs; 41 had overall score at least 0.30. Top entries were Illumina AI/iCell 0.471613, X-team/X-98 0.352404 and Vitaura/CellOS.CT 0.340544. The cluster is real and makes a reusable approach plausible; it does not establish a publicly released 0.30 implementation. Exact leading submission methods were not found in the reviewed sources. Arc explicitly [allows teams' own datasets](https://arcinstitute.org/news/virtual-cell-challenge-2026), so unequal training resources remain possible; there is no evidence here that any particular leader used private data.

Public, author-documented results found: [Sidechain](https://github.com/saberhq/sidechain) 0.1875, [AtlasShift](https://github.com/kaipengm2/Virtual-Cell-Challenge-2026) 0.154562, and [rpinho/vcc2026](https://github.com/rpinho/vcc2026/blob/main/OFFICIAL_SCORES.md) 0.1528. These are useful controls, not demonstrated 0.30 solutions. The [dossier's 0.341 scenario](https://github.com/danielchen26/vcc2026-dossier) is conditional arithmetic, not an official achieved submission score.

1. **Separate pooled and per-cell normalized means in the generator.** AtlasShift explicitly matches both. They differ because `sum_i X_ig / sum_i L_i` is not `mean_i(X_ig/L_i)`. Our totals correction alone controls the first. Fit both requested summaries and then assess the actual scorer, including the bounded variance correction and Wilcoxon ranks; two means alone do not determine all six metrics.
2. **Control detectability through variance, without automatically inflating effect sizes.** [Sidechain's model notes](https://saberhq.com/sidechain/models/) describe tightening selected genes' cell-to-cell spread while holding totals fixed. They also report an isolated change to bulk response strength raising scaled MSE from 0.143 to 0.212 while the other five metrics held. These are specific public precedents for replacing our MSE-versus-floor tradeoff, not evidence that their method reaches 0.30.
3. **Match the validation perturbation panel and available evidence.** The coverage audit above finds HepG2/RPE1/Jurkat target overlap with ABC equal to zero, although those sources cover H1 targets. Use challenge-like target coverage and leave entire studies/families out when testing transfers, including the related CD4 states. Direct target evidence from genome-wide K562, HCT116, HEK293T and CD4 deserves an explicit baseline. We already possess those sources; source integration and validation deserve attention before assuming a missing dataset.
4. **Repair shared-response estimation assumptions before interpreting Model 7.** `m7_tables.py:52` averages over each source's own retained target panel. Essential-only and genome-wide panel means conflate target mix with cell context. Estimate panel-matched means or average predicted per-target effects over the requested panel. `m7_fit.py:48` divides summed delta variances by K squared even though all deltas share a control estimate. Under independent perturbation estimates and one shared control, the mean variance is `sum(varS)/K**2 + var0`, not `sum(varS + var0)/K**2`. Missing genes need gene-specific denominators. These are code-level concerns, not measured explanations of the entire gap. No local `step1.json`/Step 1b result was available, so no Model 7 outcome is claimed.

The practical next benchmark is public atlas transfer with two-summary count emission, evaluated under the same exact scorer and panel-matched holdouts. It gives a verified public reference before another architecture search. A 0.45–0.50 scaled MSE on the current H1 anchors would require raw MSE about 0.525–0.478; the demonstrated 0.932 fix is an initial threshold crossing, not evidence that the larger gap is closed.

In [ ]:
# Small frozen public snapshot; no participant contact information retained.
leaderboard_checked_utc_date = '2026-10-21'
leaderboard_snapshot = [{'team_name': 'Illumina AI', 'model_name': 'iCell', 'score_avg': 0.4716128766836419, 'score_pds': 0.8773751094398947, 'score_mse': 0.7926135489945971, 'score_nmae': 0.43206120100211276, 'score_fid': 0.1806949445951185, 'score_reach': 0.4669712743622961, 'score_jac': 0.07996118170783202}, {'team_name': 'X-team', 'model_name': 'X-98', 'score_avg': 0.3524039185454437, 'score_pds': 0.7732659207516632, 'score_mse': 0.5079757710643764, 'score_nmae': 0.29505784701465604, 'score_fid': 0.06768533875064743, 'score_reach': 0.3784256473547421, 'score_jac': 0.09201298633657684}, {'team_name': 'Vitaura Team', 'model_name': 'CellOS.CT', 'score_avg': 0.3405439079469676, 'score_pds': 0.8469410071132047, 'score_mse': 0.5177931510155891, 'score_nmae': 0.2823142678887081, 'score_fid': 0.08334587984802118, 'score_reach': 0.2938924891788527, 'score_jac': 0.01897665263742977}, {'team_name': 'yushan liu', 'model_name': 'CellMelody', 'score_avg': 0.337000793578251, 'score_pds': 0.8514540915571688, 'score_mse': 0.4913968868497623, 'score_nmae': 0.28894121672068757, 'score_fid': 0.0837794862354881, 'score_reach': 0.2857063918395039, 'score_jac': 0.02072668826689553}, {'team_name': 'cqawesome', 'model_name': 'm1024c', 'score_avg': 0.33207521506305215, 'score_pds': 0.8314996065852954, 'score_mse': 0.46153202261089765, 'score_nmae': 0.28848616182639064, 'score_fid': 0.08898211734811706, 'score_reach': 0.29624695000063467, 'score_jac': 0.025704432006977255}, {'team_name': 'FlowState', 'model_name': 'flowstate', 'score_avg': 0.3314854864442294, 'score_pds': 0.8152786534848477, 'score_mse': 0.512408804715605, 'score_nmae': 0.2597930401507676, 'score_fid': 0.0906818073875462, 'score_reach': 0.3069409119350594, 'score_jac': 0.003809700991550773}, {'team_name': 'realcell', 'model_name': 'test', 'score_avg': 0.3295429648015781, 'score_pds': 0.8384491546892611, 'score_mse': 0.5122924609138091, 'score_nmae': 0.23994833074849894, 'score_fid': 0.07222109563027622, 'score_reach': 0.31069171686462055, 'score_jac': 0.0036550299630026197}, {'team_name': 'Junru Jin', 'model_name': 'j99rkpd', 'score_avg': 0.32676884382970695, 'score_pds': 0.8310991683158715, 'score_mse': 0.5056470334589818, 'score_nmae': 0.25361890076223365, 'score_fid': 0.053828923649917405, 'score_reach': 0.3010862199358878, 'score_jac': 0.015332816855349799}, {'team_name': 'TAO', 'model_name': 'v72', 'score_avg': 0.326720178571405, 'score_pds': 0.8300833197390703, 'score_mse': 0.49947029155695916, 'score_nmae': 0.2415115423160017, 'score_fid': 0.05799489713421805, 'score_reach': 0.31460865561779944, 'score_jac': 0.016652365064381405}, {'team_name': 'Chenhao Tan', 'model_name': 'j100small', 'score_avg': 0.3235790720991277, 'score_pds': 0.8293782385088071, 'score_mse': 0.5112859267579787, 'score_nmae': 0.2569222325756709, 'score_fid': 0.053008775282442556, 'score_reach': 0.27620019823818004, 'score_jac': 0.014679061231687081}, {'team_name': 'Aginglab.com', 'model_name': 'VirtualOmics', 'score_avg': 0.31932996091161386, 'score_pds': 0.8232236961337479, 'score_mse': 0.5209953354105591, 'score_nmae': 0.25476533004250074, 'score_fid': 0.015705006185374768, 'score_reach': 0.28285524571978593, 'score_jac': 0.018435151977714993}, {'team_name': 'C-team', 'model_name': 'The_Way55', 'score_avg': 0.31793350714242563, 'score_pds': 0.8410585597615134, 'score_mse': 0.47443319549548796, 'score_nmae': 0.20560685019655678, 'score_fid': 0.09394992778021927, 'score_reach': 0.27244512262313986, 'score_jac': 0.020107386997636385}, {'team_name': 'wllll', 'model_name': 'Run-25', 'score_avg': 0.3172341698177004, 'score_pds': 0.8328999736252213, 'score_mse': 0.4882529165800169, 'score_nmae': 0.23864041303947983, 'score_fid': 0.04917749143407799, 'score_reach': 0.27650546964218464, 'score_jac': 0.01792875458522209}, {'team_name': 'Zhengming Zhang', 'model_name': 'Run-26', 'score_avg': 0.31716318970120194, 'score_pds': 0.8328999736252213, 'score_mse': 0.4882529165800169, 'score_nmae': 0.23863506043319638, 'score_fid': 0.04924520743978128, 'score_reach': 0.27601515260179177, 'score_jac': 0.017930827527204013}, {'team_name': 'CellWM', 'model_name': 'CellWM-101', 'score_avg': 0.3158089249915843, 'score_pds': 0.7533432976892612, 'score_mse': 0.4763758552311781, 'score_nmae': 0.2847957569277389, 'score_fid': 0.07217694945992903, 'score_reach': 0.2776309332607158, 'score_jac': 0.030530757380682606}, {'team_name': 'WCK', 'model_name': 'genoguessr-v24', 'score_avg': 0.31558596752658574, 'score_pds': 0.8306599614152601, 'score_mse': 0.468653861679559, 'score_nmae': 0.2521056724208835, 'score_fid': 0.05967905387400025, 'score_reach': 0.2685305860963609, 'score_jac': 0.013886669673450583}, {'team_name': 'roxs', 'model_name': 'halcyon46', 'score_avg': 0.3154854105236931, 'score_pds': 0.8249720873314766, 'score_mse': 0.5025101587792271, 'score_nmae': 0.23515927583517857, 'score_fid': 0.05030686244913696, 'score_reach': 0.27055852012715537, 'score_jac': 0.009405558619984216}, {'team_name': 'chill', 'model_name': 'Stranded', 'score_avg': 0.3139795527120876, 'score_pds': 0.8146469154908726, 'score_mse': 0.4652201723455109, 'score_nmae': 0.26583337911505495, 'score_fid': 0.04875177781365081, 'score_reach': 0.25504805678852344, 'score_jac': 0.034377014718912906}, {'team_name': 'TOBE', 'model_name': '261021_v1', 'score_avg': 0.31173200023143416, 'score_pds': 0.8119528144021947, 'score_mse': 0.4669606636221335, 'score_nmae': 0.26927559138096097, 'score_fid': 0.038767015693944015, 'score_reach': 0.2706107185499694, 'score_jac': 0.012825197739402315}, {'team_name': 'CYH', 'model_name': 'Cyan', 'score_avg': 0.3105686896233683, 'score_pds': 0.8241829380476936, 'score_mse': 0.4514233960568355, 'score_nmae': 0.25374045281972607, 'score_fid': 0.042683628887692115, 'score_reach': 0.2763298949022327, 'score_jac': 0.015051827026029951}, {'team_name': 'Lei Huang', 'model_name': 'C29', 'score_avg': 0.31052407063132087, 'score_pds': 0.8090973755170551, 'score_mse': 0.4695932939933498, 'score_nmae': 0.2697502288511618, 'score_fid': 0.04108987457133865, 'score_reach': 0.26306565359853545, 'score_jac': 0.010547997256484547}, {'team_name': 'kk123', 'model_name': 'V8', 'score_avg': 0.30979383489981643, 'score_pds': 0.7891760070098047, 'score_mse': 0.4532981929723719, 'score_nmae': 0.238493416116728, 'score_fid': 0.08557853331719746, 'score_reach': 0.26634732132178546, 'score_jac': 0.025869538661011174}, {'team_name': 'Trans_or_Cis', 'model_name': 'ABC_final', 'score_avg': 0.3096900089700994, 'score_pds': 0.8112734016281267, 'score_mse': 0.4427001842230554, 'score_nmae': 0.24046879562107368, 'score_fid': 0.04627029865415805, 'score_reach': 0.30876420929888304, 'score_jac': 0.008663164395299855}, {'team_name': 'Ling yao Zeng', 'model_name': 'fa9fe164-545f-48ef-965f-a39dc9137f29', 'score_avg': 0.30961936269403745, 'score_pds': 0.8320824290594379, 'score_mse': 0.44279825681216917, 'score_nmae': 0.23617506278396694, 'score_fid': 0.055774349302538447, 'score_reach': 0.2671239772742147, 'score_jac': 0.02376210093189757}, {'team_name': 'Layne Wong', 'model_name': 'vcc_model_132', 'score_avg': 0.3095498802102892, 'score_pds': 0.8107630533691866, 'score_mse': 0.4694594401873304, 'score_nmae': 0.26719290633397375, 'score_fid': 0.03731830531387041, 'score_reach': 0.26076832948541584, 'score_jac': 0.011797246571958043}]
display(pd.DataFrame(leaderboard_snapshot))
